In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from statsmodels.tsa.stattools import adfuller
from sklearn.model_selection import train_test_split, GridSearchCV, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [5]:
# ============================================================
# STEP 2: LOAD THE DATASET
# ------------------------------------------------------------
# WHY THIS STEP?
# In Step 1 we imported all the tools (libraries).
# Now we need the actual DATA to work with.
# Without data, there is nothing to train the model on.
# We load the same CSV file we used in the DT notebook
# so our KNN results can be compared fairly with DT results.
# ------------------------------------------------------------
# WHAT WE ARE DOING HERE?
# - Reading the CSV file into a pandas DataFrame
# - Setting the Date column as the index
# - Checking shape (rows and columns)
# - Checking date range (start date to end date)
# - Checking null values (missing data)
# - Checking duplicates
# ============================================================

ALLData = pd.read_csv("All_features.csv", index_col="Date", parse_dates=["Date"]).dropna()

# Check basic info
print("Shape:", ALLData.shape)
print("\nDate Range:", ALLData.index.min(), "to", ALLData.index.max())
print("\nNull Count:")
print(ALLData.isnull().sum())
print("\nDuplicates:", ALLData.index.duplicated().sum())

Shape: (2342, 17)

Date Range: 2015-01-05 00:00:00 to 2025-12-30 00:00:00

Null Count:
HDFCBANK     0
NIFTY50      0
BANKNIFTY    0
SENSEX       0
INDIA_VIX    0
USDINR       0
SP500        0
NASDAQ100    0
DOWJONES     0
NIKKEI225    0
HANGSENG     0
GOLD         0
BRENT_OIL    0
ICICIBANK    0
SBIN         0
AXISBANK     0
KOTAKBANK    0
dtype: int64

Duplicates: 0


In [7]:
# ============================================================
# STEP 3: CHECK STATIONARITY ON RAW PRICES
# ------------------------------------------------------------
# WHY THIS STEP?
# In Step 2 we loaded raw price data.
# Raw prices are NOT stationary — they keep going up or down
# over time (trending). ML models like KNN assume the data
# has a stable statistical pattern (mean, variance constant).
# So before we feed data to KNN, we MUST check stationarity.
# We use the ADF test — if p-value > 0.05, NOT stationary.
# ------------------------------------------------------------
# WHAT WE ARE DOING HERE?
# - Running ADF (Augmented Dickey-Fuller) test on each column
# - If p-value > 0.05 → NOT STATIONARY (needs transformation)
# - If p-value < 0.05 → STATIONARY (ready to use)
# - We expect raw prices to FAIL this test
# ============================================================

print("=" * 45)
print("STATIONARITY CHECK — RAW PRICES")
print("=" * 45)

all_stationary = True

for col in ALLData.columns:
    result = adfuller(ALLData[col].dropna())
    status = "STATIONARY" if result[1] < 0.05 else "NOT STATIONARY"
    if result[1] >= 0.05:
        all_stationary = False
    print(f"{col:<15} p-value: {result[1]:.4f}   {status}")

print()
if all_stationary:
    print("All assets are stationary — rare for raw prices!")
else:
    print("Most assets are NOT stationary — we will fix this in Step 4 using Log Returns.")

STATIONARITY CHECK — RAW PRICES
HDFCBANK        p-value: 0.8291   NOT STATIONARY
NIFTY50         p-value: 0.1846   NOT STATIONARY
BANKNIFTY       p-value: 1.0000   NOT STATIONARY
SENSEX          p-value: 0.8286   NOT STATIONARY
INDIA_VIX       p-value: 0.9845   NOT STATIONARY
USDINR          p-value: 0.9559   NOT STATIONARY
SP500           p-value: 0.6854   NOT STATIONARY
NASDAQ100       p-value: 0.9937   NOT STATIONARY
DOWJONES        p-value: 0.9793   NOT STATIONARY
NIKKEI225       p-value: 0.9831   NOT STATIONARY
HANGSENG        p-value: 0.9947   NOT STATIONARY
GOLD            p-value: 0.2712   NOT STATIONARY
BRENT_OIL       p-value: 0.0000   STATIONARY
ICICIBANK       p-value: 0.9943   NOT STATIONARY
SBIN            p-value: 0.9963   NOT STATIONARY
AXISBANK        p-value: 0.9694   NOT STATIONARY
KOTAKBANK       p-value: 0.9820   NOT STATIONARY

Most assets are NOT stationary — we will fix this in Step 4 using Log Returns.


In [9]:
# ============================================================
# STEP 4: CONVERT RAW PRICES TO LOG RETURNS
# ------------------------------------------------------------
# WHY THIS STEP?
# In Step 3 we proved that raw prices are NOT stationary.
# KNN uses distance between data points to make predictions.
# If data is non-stationary (always trending up/down),
# the distances become meaningless and the model gets confused.
# Log Returns fix this — they represent % change day by day,
# which is stationary (no long-term trend).
# Formula: log_return = log(today_price / yesterday_price)
# ------------------------------------------------------------
# WHAT WE ARE DOING HERE?
# - Converting all 17 price columns into log returns
# - Dropping the first row (it becomes NaN after the shift)
# - Printing the new shape and previewing the data
# ============================================================

# Log return = log(today / yesterday)
log_returns = np.log(ALLData / ALLData.shift(1)).dropna()

print("Log Returns Shape:", log_returns.shape)
print()
print(log_returns.head())

Log Returns Shape: (2341, 17)

            HDFCBANK   NIFTY50  BANKNIFTY    SENSEX  INDIA_VIX    USDINR  \
Date                                                                       
2015-01-06 -0.036400 -0.038581   0.012711 -0.015689  -0.043344  0.000948   
2015-01-07 -0.000802  0.000978  -0.007161  0.002914  -0.027411  0.003625   
2015-01-08  0.006596 -0.003722  -0.001819  0.020788   0.026836 -0.004652   
2015-01-09 -0.014044 -0.016820   0.006270  0.011131  -0.016249 -0.009608   
2015-01-13  0.019111 -0.072835   0.014937 -0.012687  -0.002783 -0.009620   

               SP500  NASDAQ100  DOWJONES  NIKKEI225  HANGSENG      GOLD  \
Date                                                                       
2015-01-06 -0.011995  -0.041955 -0.031185  -0.007456 -0.008933 -0.009995   
2015-01-07  0.015974   0.000833 -0.002918   0.012180  0.011563  0.008305   
2015-01-08  0.052957   0.015538  0.013506   0.018221  0.017730  0.006493   
2015-01-09  0.015028  -0.005427  0.006712  -0.009567 -0.

In [11]:
# ============================================================
# STEP 5: CHECK STATIONARITY AGAIN ON LOG RETURNS
# ------------------------------------------------------------
# WHY THIS STEP?
# In Step 4 we converted prices into log returns.
# Now we need to verify that the transformation worked.
# If the log returns are stationary, we can safely use them
# for feature engineering and model building.
# ------------------------------------------------------------
# WHAT WE ARE DOING HERE?
# - Running ADF test again, but now on log returns
# - Confirming that the transformed data is stationary
# - This gives us a green light for the next feature step
# ============================================================

print("=" * 45)
print("STATIONARITY CHECK — LOG RETURNS")
print("=" * 45)

all_stationary = True

for col in log_returns.columns:
    result = adfuller(log_returns[col].dropna())
    status = "STATIONARY" if result[1] < 0.05 else "NOT STATIONARY"
    if result[1] >= 0.05:
        all_stationary = False
    print(f"{col:<15} p-value: {result[1]:.4f}   {status}")

print()
if all_stationary:
    print("All assets are stationary after log returns!")
else:
    print("Some assets are still not stationary — check the results above.")

STATIONARITY CHECK — LOG RETURNS
HDFCBANK        p-value: 0.0000   STATIONARY
NIFTY50         p-value: 0.0000   STATIONARY
BANKNIFTY       p-value: 0.0000   STATIONARY
SENSEX          p-value: 0.0000   STATIONARY
INDIA_VIX       p-value: 0.0000   STATIONARY
USDINR          p-value: 0.0000   STATIONARY
SP500           p-value: 0.0000   STATIONARY
NASDAQ100       p-value: 0.0000   STATIONARY
DOWJONES        p-value: 0.0000   STATIONARY
NIKKEI225       p-value: 0.0000   STATIONARY
HANGSENG        p-value: 0.0000   STATIONARY
GOLD            p-value: 0.0000   STATIONARY
BRENT_OIL       p-value: 0.0000   STATIONARY
ICICIBANK       p-value: 0.0000   STATIONARY
SBIN            p-value: 0.0000   STATIONARY
AXISBANK        p-value: 0.0000   STATIONARY
KOTAKBANK       p-value: 0.0000   STATIONARY

All assets are stationary after log returns!


In [13]:
# ============================================================
# STEP 6: CREATE FEATURES
# ------------------------------------------------------------
# WHY THIS STEP?
# In Step 5 we confirmed the data is stationary after log returns.
# Now we can create useful features from the stationary series.
# These features help KNN find patterns using distance.
# KNN works better when we give it multiple informative inputs
# instead of only raw log returns.
# ------------------------------------------------------------
# WHAT WE ARE DOING HERE?
# - Starting with a copy of the log-return data
# - Adding lag features for HDFC
# - Adding moving average features
# - Adding EMA features
# - Adding momentum feature
# - Keeping all other market return columns as base features
# ============================================================

features_df = log_returns.copy()

# --- HDFC lag features ---
# These capture how HDFC behaved in the past.
features_df["HDFClag1"] = log_returns["HDFCBANK"].shift(1)
features_df["HDFClag5"] = log_returns["HDFCBANK"].shift(5)
features_df["HDFClag10"] = log_returns["HDFCBANK"].shift(10)

# --- SMA features ---
# These smooth the past HDFC returns to show trend direction.
features_df["HDFCSMA10"] = log_returns["HDFCBANK"].shift(1).rolling(window=10).mean()
features_df["HDFCSMA50"] = log_returns["HDFCBANK"].shift(1).rolling(window=50).mean()
features_df["HDFCSMA200"] = log_returns["HDFCBANK"].shift(1).rolling(window=200).mean()

# --- EMA features ---
# These give more weight to recent HDFC values.
features_df["HDFCEMA10"] = log_returns["HDFCBANK"].shift(1).ewm(span=10, adjust=False).mean()
features_df["HDFCEMA50"] = log_returns["HDFCBANK"].shift(1).ewm(span=50, adjust=False).mean()
features_df["HDFCEMA200"] = log_returns["HDFCBANK"].shift(1).ewm(span=200, adjust=False).mean()

# --- Momentum feature ---
# This is the total HDFC return over the last 10 days.
features_df["HDFCmomentum10"] = log_returns["HDFCBANK"].shift(1).rolling(window=10).sum()

print("Features added. Shape:", features_df.shape)
print()
print(features_df[["HDFClag1", "HDFCSMA10", "HDFCEMA10", "HDFCmomentum10"]].tail())

Features added. Shape: (2341, 27)

            HDFClag1  HDFCSMA10  HDFCEMA10  HDFCmomentum10
Date                                                      
2025-12-22  0.000650  -0.004131  -0.003375       -0.041309
2025-12-23  0.002111  -0.003239  -0.002378       -0.032392
2025-12-24 -0.006672  -0.004071  -0.003159       -0.040711
2025-12-29  0.001061  -0.004176  -0.002391       -0.041764
2025-12-30  0.004637  -0.003250  -0.001113       -0.032502


In [15]:
# ============================================================
# STEP 7: CREATE THE TARGET
# ------------------------------------------------------------
# WHY THIS STEP?
# In Step 6 we created all the input features for KNN.
# Now we need the output label that KNN will try to predict.
# Since this is a classification problem, the target is:
# 1 = tomorrow's HDFC return is positive (UP)
# 0 = tomorrow's HDFC return is zero or negative (DOWN)
# ------------------------------------------------------------
# WHAT WE ARE DOING HERE?
# - Shifting HDFC log returns by -1 so today's features
#   are matched with tomorrow's movement
# - Converting the result into 1/0 classification labels
# - Adding the target column to the dataframe
# ============================================================

features_df["Target"] = (log_returns["HDFCBANK"].shift(-1) > 0).astype(int)

print("Target distribution:")
print(features_df["Target"].value_counts())
print()
print("Up days:", features_df["Target"].sum(), f"({features_df['Target'].mean()*100:.1f}%)")
print("Down days:", len(features_df) - features_df["Target"].sum(), f"({(1-features_df['Target'].mean())*100:.1f}%)")

Target distribution:
1    1212
0    1129
Name: Target, dtype: int64

Up days: 1212 (51.8%)
Down days: 1129 (48.2%)


In [17]:
# ============================================================
# STEP 8: DROP NaN VALUES
# ------------------------------------------------------------
# WHY THIS STEP?
# In Steps 6 and 7 we created lag, rolling, EMA, momentum,
# and target columns.
# These operations create missing values at the top and bottom
# of the dataframe because:
# - lag features need past values
# - rolling windows need enough observations
# - shift(-1) for target creates a missing last row
# KNN cannot handle missing values, so we must remove them.
# ------------------------------------------------------------
# WHAT WE ARE DOING HERE?
# - Dropping all rows with NaN values
# - Checking final shape after cleaning
# - Confirming how many feature columns remain
# ============================================================

features_df.dropna(inplace=True)

print("Final shape after dropping NaNs:", features_df.shape)
print("Features X columns:", features_df.shape[1] - 1)
print("Date range:", features_df.index.min(), "to", features_df.index.max())

Final shape after dropping NaNs: (2141, 28)
Features X columns: 27
Date range: 2015-12-11 00:00:00 to 2025-12-30 00:00:00


In [19]:
# ============================================================
# STEP 9: DEFINE X AND Y
# ------------------------------------------------------------
# WHY THIS STEP?
# In Step 8 we cleaned the dataset by removing all NaN rows.
# Now we split the dataframe into:
# - X = input features
# - Y = target label
# KNN will use X to learn patterns and Y as the answer it tries
# to predict.
# ------------------------------------------------------------
# WHAT WE ARE DOING HERE?
# - Dropping the Target column from the feature matrix X
# - Keeping only the Target column in Y
# - Printing shapes so we confirm both are aligned
# ============================================================

X = features_df.drop(columns=["Target"])
Y = features_df["Target"]

print("X shape:", X.shape)
print("Y shape:", Y.shape)
print()
print("Feature columns:")
for i, col in enumerate(X.columns, 1):
    print(f"{i:2d}. {col}")

X shape: (2141, 27)
Y shape: (2141,)

Feature columns:
 1. HDFCBANK
 2. NIFTY50
 3. BANKNIFTY
 4. SENSEX
 5. INDIA_VIX
 6. USDINR
 7. SP500
 8. NASDAQ100
 9. DOWJONES
10. NIKKEI225
11. HANGSENG
12. GOLD
13. BRENT_OIL
14. ICICIBANK
15. SBIN
16. AXISBANK
17. KOTAKBANK
18. HDFClag1
19. HDFClag5
20. HDFClag10
21. HDFCSMA10
22. HDFCSMA50
23. HDFCSMA200
24. HDFCEMA10
25. HDFCEMA50
26. HDFCEMA200
27. HDFCmomentum10


In [21]:
# ============================================================
# STEP 10: TRAIN-TEST SPLIT
# ------------------------------------------------------------
# WHY THIS STEP?
# Now that we have X and Y, we need to separate the data into:
# - training set: used to teach the model
# - test set: used to check how well the model learned
# Since this is time-series data, we must NOT shuffle the rows.
# We split in chronological order so the model only learns from
# the past and is tested on the future.
# ------------------------------------------------------------
# WHAT WE ARE DOING HERE?
# - Splitting X and Y into train and test sets
# - Using shuffle=False to preserve time order
# - Keeping 70% training and 30% testing (same idea as DT)
# ============================================================

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.3, shuffle=False
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Y_train shape:", Y_train.shape)
print("Y_test shape:", Y_test.shape)
print()
print("Train date range:", X_train.index.min(), "to", X_train.index.max())
print("Test date range:", X_test.index.min(), "to", X_test.index.max())

X_train shape: (1498, 27)
X_test shape: (643, 27)
Y_train shape: (1498,)
Y_test shape: (643,)

Train date range: 2015-12-11 00:00:00 to 2022-12-20 00:00:00
Test date range: 2022-12-21 00:00:00 to 2025-12-30 00:00:00


In [23]:
# ============================================================
# STEP 11: SCALE THE FEATURES
# ------------------------------------------------------------
# WHY THIS STEP?
# KNN works by measuring distance between rows.
# If one feature has large values and another has small values,
# the large-value feature can dominate the distance calculation.
# To prevent this, we standardize all features so they are on
# the same scale.
# ------------------------------------------------------------
# WHAT WE ARE DOING HERE?
# - Creating a StandardScaler
# - Fitting it only on X_train to avoid data leakage
# - Transforming both X_train and X_test
# - Keeping the scaled versions for KNN training
# ============================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled X_train shape:", X_train_scaled.shape)
print("Scaled X_test shape:", X_test_scaled.shape)

Scaled X_train shape: (1498, 27)
Scaled X_test shape: (643, 27)


In [25]:
# ============================================================
# STEP 12: CHOOSE THE KNN MODEL
# ------------------------------------------------------------
# WHY THIS STEP?
# In Step 11 we scaled the data, which is mandatory for KNN.
# Now we create the KNN classifier itself.
# KNN predicts by looking at the nearest training examples
# and letting them vote on the class.
# ------------------------------------------------------------
# WHAT WE ARE DOING HERE?
# - Creating a basic KNN classifier
# - Using a default k value first (k=5)
# - This gives us a simple baseline before tuning
# ============================================================

knn = KNeighborsClassifier(n_neighbors=5)

print(knn)

KNeighborsClassifier()


In [29]:
# ============================================================
# STEP 13: TUNE HYPERPARAMETERS
# ------------------------------------------------------------
# WHY THIS STEP?
# In Step 12 we created a basic KNN model with k=5.
# But KNN performance depends heavily on the value of k,
# and also on how neighbors are weighted.
# So now we search for the best settings using GridSearchCV.
# ------------------------------------------------------------
# WHAT WE ARE DOING HERE?
# - Testing multiple values of n_neighbors
# - Testing uniform vs distance weights
# - Using TimeSeriesSplit because this is time-series data
# - Finding the best KNN configuration from the training set
# ============================================================

param_grid = {
    "n_neighbors": [3, 5, 7, 9, 11, 13, 15],
    "weights": ["uniform", "distance"]
}

tscv = TimeSeriesSplit(n_splits=5)

grid = GridSearchCV(
    KNeighborsClassifier(),
    param_grid=param_grid,
    cv=tscv,
    scoring="accuracy",
    n_jobs=1
)

grid.fit(X_train_scaled, Y_train)

print("Best parameters:", grid.best_params_)
print("Best CV score:", grid.best_score_)

Best parameters: {'n_neighbors': 15, 'weights': 'uniform'}
Best CV score: 0.5469879518072289


In [31]:
# ============================================================
# STEP 14: FIT THE MODEL AND PREDICT
# ------------------------------------------------------------
# WHY THIS STEP?
# In Step 13 we found the best KNN settings using GridSearchCV.
# Now we use that best model to actually make predictions.
# We predict on both training and test data so we can check
# whether the model is learning properly or just memorizing.
# ------------------------------------------------------------
# WHAT WE ARE DOING HERE?
# - Taking the best KNN model from GridSearchCV
# - Predicting on training data
# - Predicting on test data
# - Saving predictions for later evaluation
# ============================================================

best_knn = grid.best_estimator_

Y_train_pred = best_knn.predict(X_train_scaled)
Y_test_pred = best_knn.predict(X_test_scaled)

print("Predictions completed.")
print("Train predictions shape:", Y_train_pred.shape)
print("Test predictions shape:", Y_test_pred.shape)

Predictions completed.
Train predictions shape: (1498,)
Test predictions shape: (643,)


In [33]:
# ============================================================
# STEP 15: CHECK ACCURACY
# ------------------------------------------------------------
# WHY THIS STEP?
# In Step 14 we generated predictions from the best KNN model.
# Now we compare those predictions with the true labels.
# Accuracy tells us the percentage of correct predictions.
# We check both train and test accuracy to see whether the
# model is learning well or just memorizing.
# ------------------------------------------------------------
# WHAT WE ARE DOING HERE?
# - Calculating training accuracy
# - Calculating test accuracy
# - Printing both values for comparison
# ============================================================

train_acc = accuracy_score(Y_train, Y_train_pred)
test_acc = accuracy_score(Y_test, Y_test_pred)

print("Train Accuracy:", train_acc)
print("Test Accuracy:", test_acc)

Train Accuracy: 0.630173564753004
Test Accuracy: 0.5194401244167963


In [35]:
# ============================================================
# STEP 16: PRINT CLASSIFICATION REPORT
# ------------------------------------------------------------
# WHY THIS STEP?
# Accuracy alone does not show the full picture.
# We also need precision, recall, and F1-score to understand
# how well KNN is predicting each class.
# This is important because the market may have slightly
# imbalanced classes, and a model can look okay on accuracy
# while still failing on one side.
# ------------------------------------------------------------
# WHAT WE ARE DOING HERE?
# - Printing classification report for train data
# - Printing classification report for test data
# - Looking at class-wise performance for 0 and 1
# ============================================================

print("TRAIN CLASSIFICATION REPORT")
print(classification_report(Y_train, Y_train_pred))

print("TEST CLASSIFICATION REPORT")
print(classification_report(Y_test, Y_test_pred))

TRAIN CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.61      0.63      0.62       710
           1       0.65      0.63      0.64       788

    accuracy                           0.63      1498
   macro avg       0.63      0.63      0.63      1498
weighted avg       0.63      0.63      0.63      1498

TEST CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.51      0.51      0.51       313
           1       0.53      0.53      0.53       330

    accuracy                           0.52       643
   macro avg       0.52      0.52      0.52       643
weighted avg       0.52      0.52      0.52       643



In [37]:
# ============================================================
# STEP 17: CONFUSION MATRIX
# ------------------------------------------------------------
# WHY THIS STEP?
# Accuracy and classification report are useful, but they don't
# show the exact count of correct and incorrect predictions.
# The confusion matrix gives a clear table of:
# - true negatives
# - false positives
# - false negatives
# - true positives
# This helps us understand where KNN is making mistakes.
# ------------------------------------------------------------
# WHAT WE ARE DOING HERE?
# - Building confusion matrix for train data
# - Building confusion matrix for test data
# - Printing both matrices
# ============================================================

print("TRAIN CONFUSION MATRIX")
print(confusion_matrix(Y_train, Y_train_pred))

print("\nTEST CONFUSION MATRIX")
print(confusion_matrix(Y_test, Y_test_pred))

TRAIN CONFUSION MATRIX
[[445 265]
 [289 499]]

TEST CONFUSION MATRIX
[[160 153]
 [156 174]]


In [39]:
# ============================================================
# STEP 18: COMPARE WITH BASELINE
# ------------------------------------------------------------
# WHY THIS STEP?
# A model is only useful if it beats a simple naive baseline.
# For this binary target, the simplest baseline is predicting
# the majority class every time.
# This tells us whether KNN is actually adding value.
# ------------------------------------------------------------
# WHAT WE ARE DOING HERE?
# - Finding the majority class in the training set
# - Predicting that class for every test observation
# - Comparing baseline accuracy with KNN test accuracy
# ============================================================

majority_class = Y_train.mode()[0]
baseline_pred = np.full_like(Y_test, fill_value=majority_class)

baseline_acc = accuracy_score(Y_test, baseline_pred)

print("Majority class:", majority_class)
print("Baseline Accuracy:", baseline_acc)
print("KNN Test Accuracy:", test_acc)

Majority class: 1
Baseline Accuracy: 0.5132192846034215
KNN Test Accuracy: 0.5194401244167963


In [41]:
# ============================================================
# STEP 19: FINAL CONCLUSION
# ------------------------------------------------------------
# WHY THIS STEP?
# After building, tuning, predicting, and evaluating the model,
# we now write the final conclusion.
# This summarizes whether KNN was useful, how it performed,
# and what we learned from the experiment.
# ------------------------------------------------------------
# WHAT WE ARE DOING HERE?
# - Comparing KNN vs baseline
# - Summarizing train/test performance
# - Writing a short conclusion for the notebook
# ============================================================

print("FINAL KNN MODEL SUMMARY")
print("------------------------")
print("Train Accuracy:", train_acc)
print("Test Accuracy:", test_acc)
print("Baseline Accuracy:", baseline_acc)

if test_acc > baseline_acc:
    print("\nConclusion: KNN is slightly better than the naive baseline, so it has some predictive value.")
else:
    print("\nConclusion: KNN does not beat the baseline, so it is not adding much value.")

print("\nNote: The train-test gap is small, so the model is not heavily overfitting, but overall performance is modest.")

FINAL KNN MODEL SUMMARY
------------------------
Train Accuracy: 0.630173564753004
Test Accuracy: 0.5194401244167963
Baseline Accuracy: 0.5132192846034215

Conclusion: KNN is slightly better than the naive baseline, so it has some predictive value.

Note: The train-test gap is small, so the model is not heavily overfitting, but overall performance is modest.


In [43]:
# KNN hyperparameter tuning is used to find the best model settings that improve test accuracy and generalization.

# n_neighbors controls how many nearby points are used for voting.
# Smaller values can make the model too sensitive to noise, while larger values can make it too smooth.

# weights decides whether each neighbor contributes equally or whether closer neighbors have more influence.
# 'distance' gives more importance to closer points, while 'uniform' treats all neighbors equally.

# metric defines how distance is calculated between data points.
# Different distance metrics can affect performance depending on the dataset structure.

# GridSearchCV is used to test multiple combinations of hyperparameters and select the best one using cross-validation.
# This helps avoid choosing parameters based only on a single train-test split.

# Feature scaling is important for KNN because it is a distance-based algorithm.
# Without scaling, features with larger numeric ranges can dominate the distance calculation.

# The tuned KNN model is then evaluated on the test set to check whether the selected parameters improve real predictive performance.